In [2]:
import numpy_financial as npf
import numpy as np
import pandas as pd
from datetime import datetime as dt, timedelta as td

def set_decimals(n, d = 2):
    return np.round(n,d)

indx = ["TIR", "Cuartil-1", "Mediana", "Cuartil-3", "Min", "Max", "Promedio", "Desv-Estandar"]  

class Portfolio:
    '''
    En Fintual, un portafolio es una combinación de instrumentos financieros en los que se invierte tu dinero, ajustada a un nivel de riesgo y 
    retorno que se alinea con tus objetivos de inversión. Cada portafolio está diseñado con una estrategia específica que incluye una mezcla 
    de activos como bonos, acciones internacionales, y otros instrumentos, lo que permite diversificar la inversión y reducir el riesgo.
    
    __ini__ (sheet_name):

        Parameters
        ----------
        sheet_name : Str
            Nombre de la pestaña del archivo Excel de donde se obtendrá los datos del Portafolio
    '''

    def __init__(self, sheet_name):
        self.name       = sheet_name                                           # Str       : Nombre del portafolio
        self.info       = pd.read_excel(f"precios.xlsx",sheet_name=sheet_name) # DataFrame : get_excel_info(sheet_name)
        self.primer_dia = self.info.iloc[-1]                                   # Series    : Datos del primer día historico del portafolio 
        self.ultimo_dia = self.info.iloc[ 0]                                   # Series    : Datos del último día del portafolio registrado en el documento
        self.anos       = self.info['ano'].unique().tolist()[::-1]             # list      : Listado de años transcurrido en el listado

        # Datos Precio del Portafolio de inicio a fin
        # Dataframe historico
        port_price = self.section_info(self.info, column_info_name="precio")

        # TIR
        self.t_tir      = port_price[0]
        # Cuartil 1
        self.t_cuatil1  = port_price[1]
        # Mediana
        self.t_mediana  = port_price[2]
        # Cuartil 3
        self.t_cuatil3  = port_price[3]
        # Min
        self.t_min      = port_price[4]
        # Max
        self.t_max      = port_price[5]
        # Promedio
        self.t_promedio = port_price[6]
        # Desviación estandar
        self.t_desv_std = port_price[7]


        # Dataframe con los datos estadisticos de "Precio" de la acción en el transcurso anual y mensual 
        self.t_price_year_info, self.t_price_months_info        = self.info_per_year(self.info, self.anos,months=True, column_info_name="precio")
        # Dataframe con los datos estadisticos de "Activos totales" de la acción en el transcurso anual y mensual
        self.t_total_assets_year_info, self.t_total_assets_months_info = self.info_per_year(self.info, self.anos,months=True, column_info_name="activos_totales")
        # Dataframe con los datos estadisticos de "Accionistas" de la acción en el transcurso anual y mensual
        self.t_shareholders_year_info, self.t_shareholders_months_info = self.info_per_year(self.info, self.anos,months=True, column_info_name="accionistas")
     

    def __str__(self):
        return self.name 

    def months_especific_info(self , df, value_name):
        return df.map(lambda x: x.get(value_name) if isinstance(x, dict) else None)
    
    # def get_months_especific_info(self, value_name):
    #     return RN_info.t_price_months_info.map(lambda x: x.get(value_name) if isinstance(x, dict) else None)

    def section_info(self, info, column_info_name="precio"):
        """Obtiene y calcula los valores estadisticos de inicio a fin de una columna en específico(Precio por defecto) del archivo de Excel 

        Parameters
        ----------
        info : Pandas DataFrame
            Dataframe con la información de Excel con los datos de la bolsa de inversión seleccionada 

        column_info_name : str , opcional
            Nombre de la columna del Dataframe a seleccionar, por defecto es "precio"

        Returns
        -------
        lista
            una lista con los valores principales de estadistica para el analisis de datos
        """
        po = info[[column_info_name]].to_numpy().squeeze()
            
        tir      = np.round(npf.irr([info.iloc[-1][column_info_name]*-1,info.iloc[0][column_info_name]])*100, decimals=2)
        cuatil1  = set_decimals(np.quantile(po, 0.25), 4)
        mediana  = set_decimals(np.quantile(po, 0.50), 4)
        cuatil3  = set_decimals(np.quantile(po, 0.75), 4)
        min      = np.rint(np.min(po))
        max      = np.rint(np.max(po))
        promedio = np.rint(np.average(po))
        desv_std = set_decimals(np.std(po),2)

        return [tir,cuatil1,mediana,cuatil3, min, max, promedio, desv_std ]


    def info_per_month(self, year_info, column_info_name="precio"):
        """ Funcion que permite adquirir los datos estadisticos a nivel mensual dentro del rango anual de un Dataframe 

        Parameters
        ----------
        year_info : Pandas DataFrame
            Dataframe con los datos anuales del protafolio

        column_info_name : str , opcional
            Nombre de la columna del Dataframe a seleccionar, por defecto es "precio"

        Returns
        -------
        Diccionario
            Diccionario que contiene el [año]: con otro diccionario que contiene los datos [estadísticos] 
        """

        tem_mensual = {}
        months_list =  year_info['mes'].unique().tolist()[::-1]
        for m in months_list:
            month_info = year_info[year_info['mes']==m]
            tem_mensual[m] = dict(zip(indx,self.section_info(month_info, column_info_name)))
        return tem_mensual


    def info_per_year(self, info, year_list, months=False, column_info_name="precio"):
        """ Funcion que permite adquirir los datos estadisticos a nivel anual dentro del rango total de un Dataframe 

        Parameters
        ----------
        year_list : lista
            listado de los años a que serán utilidazon en el iterable para los datos estadístico en el periodo de cada año

        months : Boolean
            En caso de ser 'True' tambien retornará la estadisticas a nivel menusal

        column_info_name : str , opcional
            Nombre de la columna del Dataframe a seleccionar, por defecto es "precio"
            
        Returns
        -------
        Dataframe
            DataFrame que contiene el [año]: con los datos [estadísticos] 
        
        Dataframe (Opcional)
            DataFrame que contiene el [mes]: con los datos [estadísticos] 
        
        """

        dict_anual   = {}
        dict_mensual = {}
        
        for y in year_list:
            year_info = info[info['ano']==y]
            dict_anual[y] = self.section_info(year_info, column_info_name)
            if months == True:
                dict_mensual[y] = self.info_per_month(year_info, column_info_name)
        
        df_anual  = pd.DataFrame(dict_anual, index=indx)

        if months == True:
            df_months = pd.DataFrame(dict_mensual)
            df_months = df_months.sort_index().T 
            return df_anual.T, df_months
        else:
            return df_anual.T
    

    def inf_between_2_dates(self, date1, date2, column_info_name="precio"):
        """ Función que entrega las estadistica entre dos fechas dadas(Rango inlusivo)

        Parameters
        ----------
        date1 : str
            Fecha inicial

        date2 : str
            Fecha final

        column_info_name : str , opcional
            Nombre de la columna del Dataframe a seleccionar, por defecto es "precio"
            
        Returns
        -------
        Dataframe
            Diccionario que contiene la [Fecha]: con  los datos [estadísticos] 
        """
        n1 = self.info.index[self.info['fecha'] == date1][0]
        n2 = self.info.index[self.info['fecha'] == date2][0]

        info_date_filter = self.info.iloc[n2: n1+1]

        return pd.DataFrame({f"{date1}/{date2}" : self.section_info(info_date_filter, column_info_name)}, index=indx)
        
    
    def info_per_month_from_date(self, original_date, months=1, column_info_name="precio"):
        """ Funcion que permite adquirir los datos estadisticos a nivel mensual dentro del rango anual de un Dataframe 

        Parameters
        ----------
        original_date : str
            Fecha inicial donde se calculará los meses

        months : int
            Cantidad de meses que se requiere obtener, cada mes corresponde a 30 días.
        
        column_info_name : str , opcional
            Nombre de la columna del Dataframe a seleccionar, por defecto es "precio"

        Returns
        -------
        Diccionario
            Diccionario que contiene el [año]: con otro diccionario que contiene los datos [estadísticos] 
        """
        parsed_date = dt.strptime(original_date, '%Y-%m-%d')
        li = []

        li.append(str(parsed_date)[:10])

        for i in range(1, months+1):
            li.append(str(parsed_date + td(days=30*i ))[:10])

        dt_per_month = pd.DataFrame(self.inf_between_2_dates(li[0], li[1], column_info_name))

        for i in range(1, len(li)-1):
            dt_per_month = dt_per_month.join(self.inf_between_2_dates(li[i], li[i+1], column_info_name))

        dt_per_month = dt_per_month.T

        return dt_per_month

RN_info = Portfolio('Risky Norris')
# RN_info.inf_between_2_dates('2023-02-23', '2023-08-23')
# RN_info.info
# RN_info.t_price_year_info
# RN_info.t_price_months_info

# RN_info.t_price_months_info.map(lambda x: x[0] if isinstance(x, list) else x) ###########

# RN_info.t_price_months_info.map(lambda x: x.get('TIR') if isinstance(x, dict) else None) #################################
# RN_info.t_price_months_info.map(lambda x: x.get('TIR') if isinstance(x, dict) else None)

# RN_info.months_especific_info(RN_info.t_price_months_info,'TIR')
# RN_info.months_especific_info(RN_info.t_price_months_info,'TIR')

# setattr(self.t_price_months_info, 'get_params', self._months_especific_info)

# RN_info.t_price_months_info.get_params(RN_info.t_price_months_info, "TIR")
RN_info.months_especific_info(RN_info.t_price_months_info, "TIR")


,1,2,3,4,5,6,7,8,9,10,11,12
2018,NaN,0.46,-0.44,0.93,5.75,2.16,0.78,8.77,-3.21,-1.52,-2.33,-6.70
2019,3.26,2.52,4.00,3.52,-2.28,2.01,3.88,1.20,2.46,4.61,12.78,-5.29
2020,6.82,-5.68,-8.25,14.12,1.51,6.24,-2.09,11.19,-2.93,-5.17,12.85,-2.45
2021,4.36,-3.22,-3.83,0.85,0.94,5.07,1.08,3.97,-0.05,7.38,4.08,3.50
2022,-13.11,-3.66,1.35,-3.07,-2.47,4.15,5.25,-4.41,-2.94,2.64,0.98,-9.59
2023,2.74,1.63,1.70,0.29,4.48,4.59,8.39,-1.74,-0.51,-3.71,6.70,6.73
2024,5.34,7.99,2.43,-6.70,1.20,7.07,-0.14,0.01,NaN,NaN,NaN,NaN


In [3]:
exc = pd.read_excel(f"fechas_importantes.xlsx",sheet_name="Covid-19")
exc = exc[["Fecha de inicio", "Fecha de término"]]


li = []

for index, row in exc.iterrows():
    ini = str(row['Fecha de inicio'])[0:10]
    fin = str(row['Fecha de término'])[0:10]
    li.append(RN_info.inf_between_2_dates(ini, fin).loc['TIR'].iloc[0])



print(li)


[6.81, -5.68, -8.25, 22.34, 12.3, 2.4, 26.74, -15.93]


In [6]:
all_dates = {}
xls = pd.read_excel(f"fechas_importantes.xlsx",sheet_name=None)
sheet_names = list(xls.keys())
for p in sheet_names:
    li = []
    for index, row in exc.iterrows():
        ini = str(row['Fecha de inicio'])[0:10]
        fin = str(row['Fecha de término'])[0:10]
        li.append(RN_info.inf_between_2_dates(ini, fin).loc['TIR'].iloc[0])
    all_dates[p] = li

for idx, i in all_dates.items():
    print(f"{idx} - {i}")

Covid-19 - [6.81, -5.68, -8.25, 22.34, 12.3, 2.4, 26.74, -15.93]
EEUU vs China - [6.81, -5.68, -8.25, 22.34, 12.3, 2.4, 26.74, -15.93]
Guerra en Ucrania - [6.81, -5.68, -8.25, 22.34, 12.3, 2.4, 26.74, -15.93]


In [11]:
# Obtener los nombres de todas las pestañas (u hojas) del archivo Excel
def get_all_portfolios_info():
    all_portfolios = {}
    xls = pd.read_excel(f"precios.xlsx",sheet_name=None)
    sheet_names = list(xls.keys())
    print(sheet_names)
    for p in sheet_names:
        tem_port = Portfolio(p)
        all_portfolios[tem_port.name] = tem_port
    
    return all_portfolios

apt = get_all_portfolios_info()


apt['Risky Norris'].t_price_year_info


['Risky Norris', 'Moderate Pitt', 'Conservative Clooney']


,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2018,4.15,1035.5622,1097.5006,1138.3369,974.0,1227.0,1092.0,65.38
2019,37.52,1158.9678,1218.9270,1285.2729,1036.0,1552.0,1235.0,116.52
2020,23.09,1476.0929,1564.6428,1684.7567,1175.0,1814.0,1574.0,140.50
2021,31.16,1855.1156,1980.1353,2096.9327,1712.0,2375.0,2002.0,175.00
2022,-26.48,1841.3884,1899.6880,1970.6953,1681.0,2352.0,1915.0,112.91
2023,32.61,1771.2368,1944.6561,2088.1245,1684.0,2302.0,1943.0,168.73
2024,16.52,2519.0018,2579.1106,2674.9374,2211.0,2784.0,2571.0,127.55


In [28]:
RN_info.t_price_months_info.map(lambda x: x.get('TIR') if isinstance(x, dict) else None)

,1,2,3,4,5,6,7,8,9,10,11,12
2018,NaN,0.46,-0.44,0.93,5.75,2.16,0.78,8.77,-3.21,-1.52,-2.33,-6.70
2019,3.26,2.52,4.00,3.52,-2.28,2.01,3.88,1.20,2.46,4.61,12.78,-5.29
2020,6.82,-5.68,-8.25,14.12,1.51,6.24,-2.09,11.19,-2.93,-5.17,12.85,-2.45
2021,4.36,-3.22,-3.83,0.85,0.94,5.07,1.08,3.97,-0.05,7.38,4.08,3.50
2022,-13.11,-3.66,1.35,-3.07,-2.47,4.15,5.25,-4.41,-2.94,2.64,0.98,-9.59
2023,2.74,1.63,1.70,0.29,4.48,4.59,8.39,-1.74,-0.51,-3.71,6.70,6.73
2024,5.34,7.99,2.43,-6.70,1.20,7.07,-0.14,0.01,NaN,NaN,NaN,NaN


In [27]:
RN_info.t_price_months_info

,1,2,3,4,5,6,7,8,9,10,11,12
2018,NaN,"{'TIR': 0.46, 'Cuartil-1': 1009.1572, 'Mediana...","{'TIR': -0.44, 'Cuartil-1': 998.9066, 'Mediana...","{'TIR': 0.93, 'Cuartil-1': 989.7455, 'Mediana'...","{'TIR': 5.75, 'Cuartil-1': 1046.7514, 'Mediana...","{'TIR': 2.16, 'Cuartil-1': 1086.0065, 'Mediana...","{'TIR': 0.78, 'Cuartil-1': 1115.114, 'Mediana'...","{'TIR': 8.77, 'Cuartil-1': 1135.9855, 'Mediana...","{'TIR': -3.21, 'Cuartil-1': 1185.3523, 'Median...","{'TIR': -1.52, 'Cuartil-1': 1131.6108, 'Median...","{'TIR': -2.33, 'Cuartil-1': 1096.8821, 'Median...","{'TIR': -6.7, 'Cuartil-1': 1045.28, 'Mediana':..."
2019,"{'TIR': 3.26, 'Cuartil-1': 1060.6201, 'Mediana...","{'TIR': 2.52, 'Cuartil-1': 1081.168, 'Mediana'...","{'TIR': 4.0, 'Cuartil-1': 1116.7547, 'Mediana'...","{'TIR': 3.52, 'Cuartil-1': 1160.7664, 'Mediana...","{'TIR': -2.28, 'Cuartil-1': 1187.5384, 'Median...","{'TIR': 2.01, 'Cuartil-1': 1193.5206, 'Mediana...","{'TIR': 3.88, 'Cuartil-1': 1228.3532, 'Mediana...","{'TIR': 1.2, 'Cuartil-1': 1235.1242, 'Mediana'...","{'TIR': 2.46, 'Cuartil-1': 1278.7368, 'Mediana...","{'TIR': 4.61, 'Cuartil-1': 1271.7176, 'Mediana...","{'TIR': 12.78, 'Cuartil-1': 1385.0471, 'Median...","{'TIR': -5.29, 'Cuartil-1': 1439.5355, 'Median..."
2020,"{'TIR': 6.82, 'Cuartil-1': 1491.3702, 'Mediana...","{'TIR': -5.68, 'Cuartil-1': 1535.7097, 'Median...","{'TIR': -8.25, 'Cuartil-1': 1263.0402, 'Median...","{'TIR': 14.12, 'Cuartil-1': 1396.0648, 'Median...","{'TIR': 1.51, 'Cuartil-1': 1449.7446, 'Mediana...","{'TIR': 6.24, 'Cuartil-1': 1482.7594, 'Mediana...","{'TIR': -2.09, 'Cuartil-1': 1545.4505, 'Median...","{'TIR': 11.19, 'Cuartil-1': 1633.7058, 'Median...","{'TIR': -2.93, 'Cuartil-1': 1605.0, 'Mediana':...","{'TIR': -5.17, 'Cuartil-1': 1681.4424, 'Median...","{'TIR': 12.85, 'Cuartil-1': 1705.474, 'Mediana...","{'TIR': -2.45, 'Cuartil-1': 1769.824, 'Mediana..."
2021,"{'TIR': 4.36, 'Cuartil-1': 1836.9556, 'Mediana...","{'TIR': -3.22, 'Cuartil-1': 1880.7804, 'Median...","{'TIR': -3.83, 'Cuartil-1': 1799.8378, 'Median...","{'TIR': 0.85, 'Cuartil-1': 1832.9915, 'Mediana...","{'TIR': 0.94, 'Cuartil-1': 1767.9376, 'Mediana...","{'TIR': 5.07, 'Cuartil-1': 1866.9084, 'Mediana...","{'TIR': 1.08, 'Cuartil-1': 1980.1802, 'Mediana...","{'TIR': 3.97, 'Cuartil-1': 2046.3602, 'Mediana...","{'TIR': -0.05, 'Cuartil-1': 2075.8462, 'Median...","{'TIR': 7.38, 'Cuartil-1': 2108.044, 'Mediana'...","{'TIR': 4.08, 'Cuartil-1': 2274.8134, 'Mediana...","{'TIR': 3.5, 'Cuartil-1': 2266.7667, 'Mediana'..."
2022,"{'TIR': -13.11, 'Cuartil-1': 1977.6166, 'Media...","{'TIR': -3.66, 'Cuartil-1': 1944.9289, 'Median...","{'TIR': 1.35, 'Cuartil-1': 1887.1906, 'Mediana...","{'TIR': -3.07, 'Cuartil-1': 1894.8848, 'Median...","{'TIR': -2.47, 'Cuartil-1': 1794.5606, 'Median...","{'TIR': 4.15, 'Cuartil-1': 1760.17, 'Mediana':...","{'TIR': 5.25, 'Cuartil-1': 1970.4219, 'Mediana...","{'TIR': -4.41, 'Cuartil-1': 1985.0313, 'Median...","{'TIR': -2.94, 'Cuartil-1': 1854.0436, 'Median...","{'TIR': 2.64, 'Cuartil-1': 1810.6392, 'Mediana...","{'TIR': 0.98, 'Cuartil-1': 1825.2305, 'Mediana...","{'TIR': -9.59, 'Cuartil-1': 1747.1044, 'Median..."
2023,"{'TIR': 2.74, 'Cuartil-1': 1715.3646, 'Mediana...","{'TIR': 1.63, 'Cuartil-1': 1748.7266, 'Mediana...","{'TIR': 1.7, 'Cuartil-1': 1733.2664, 'Mediana'...","{'TIR': 0.29, 'Cuartil-1': 1777.0128, 'Mediana...","{'TIR': 4.48, 'Cuartil-1': 1763.8436, 'Mediana...","{'TIR': 4.59, 'Cuartil-1': 1873.8136, 'Mediana...","{'TIR': 8.39, 'Cuartil-1': 1952.713, 'Mediana'...","{'TIR': -1.74, 'Cuartil-1': 2029.7378, 'Median...","{'TIR': -0.51, 'Cuartil-1': 2067.8996, 'Median...","{'TIR': -3.71, 'Cuartil-1': 2067.2061, 'Median...","{'TIR': 6.7, 'Cuartil-1': 2084.3149, 'Mediana'...","{'TIR': 6.73, 'Cuartil-1': 2130.8436, 'Mediana..."
2024,"{'TIR': 5.34, 'Cuartil-1': 2292.203, 'Mediana'...","{'TIR': 7.99, 'Cuartil-1': 2521.6709, 'Mediana...","{'TIR': 2.43, 'Cuartil-1': 2611.9736, 'Mediana...","{'TIR': -6.7, 'Cuartil-1': 2514.7873, 'Mediana...","{'TIR': 1.2, 'Cuartil-1': 2512.8

In [29]:
RN_info.t_price_year_info

,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2018,4.15,1035.5622,1097.5006,1138.3369,974.0,1227.0,1092.0,65.38
2019,37.52,1158.9678,1218.9270,1285.2729,1036.0,1552.0,1235.0,116.52
2020,23.09,1476.0929,1564.6428,1684.7567,1175.0,1814.0,1574.0,140.50
2021,31.16,1855.1156,1980.1353,2096.9327,1712.0,2375.0,2002.0,175.00
2022,-26.48,1841.3884,1899.6880,1970.6953,1681.0,2352.0,1915.0,112.91
2023,32.61,1771.2368,1944.6561,2088.1245,1684.0,2302.0,1943.0,168.73
2024,16.52,2519.0018,2579.1106,2674.9374,2211.0,2784.0,2571.0,127.55


In [14]:
print(RN_info.t_tir)

162.67


In [15]:
RN_info.t_shareholders_year_info

,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2018,105700.00,79.00,340.0,896.0,1.0,1058.0,458.0,393.25
2019,346.41,1495.00,2233.0,3298.0,1058.0,4723.0,2443.0,1068.13
2020,370.53,6742.25,8765.0,17311.0,4723.0,22223.0,11572.0,5659.00
2021,169.40,30045.00,36518.0,45202.0,22223.0,59869.0,38193.0,9963.39
2022,-14.32,54648.00,57442.0,62424.0,51296.0,63877.0,57989.0,4015.37
2023,-10.59,46038.00,46285.0,48501.0,0.0,51296.0,46172.0,7387.64
2024,16.85,49667.00,51820.0,52648.0,45864.0,53824.0,51062.0,2299.19


In [16]:
RN_info.t_total_assets_year_info

,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2018,183362.63,1.319391e+08,4.093783e+08,1.216643e+09,7.083570e+05,1.903941e+09,5.949498e+08,5.091385e+08
2019,871.72,2.099611e+09,4.166821e+09,6.253056e+09,1.299570e+09,1.365892e+10,4.938535e+09,3.302394e+09
2020,563.54,2.114669e+10,2.888917e+10,5.686275e+10,1.262815e+10,8.379329e+10,3.837453e+10,2.118402e+10
2021,268.12,1.177654e+11,1.547051e+11,2.102116e+11,8.205475e+10,3.171426e+11,1.713907e+11,6.270201e+10
2022,-35.85,2.284893e+11,2.497248e+11,2.660274e+11,1.946809e+11,3.313304e+11,2.495676e+11,2.558688e+10
2023,21.72,2.008146e+11,2.076095e+11,2.212519e+11,1.902856e+11,2.415882e+11,2.109110e+11,1.180456e+10
2024,33.88,2.932620e+11,3.018919e+11,3.210975e+11,2.359821e+11,3.414285e+11,3.007588e+11,2.526153e+10


In [17]:
RN_info.info_per_month_from_date('2023-02-23', 3)

,TIR,Cuartil-1,Mediana,Cuartil-3,Min,Max,Promedio,Desv-Estandar
2023-02-23/2023-03-25,1.49,1735.0282,1762.1509,1779.0434,1684.0,1794.0,1752.0,32.39
2023-03-25/2023-04-24,2.10,1767.6624,1781.3862,1801.7927,1737.0,1827.0,1783.0,20.53
2023-04-24/2023-05-24,0.33,1760.1441,1773.2863,1797.6408,1746.0,1830.0,1778.0,24.41


In [18]:
print(RN_info.info)

      Unnamed: 0       fecha   ano  mes  dia     precio  accionistas  \
0              1  2024-08-27  2024    8   27  2636.8129        53591   
1              2  2024-08-26  2024    8   26  2631.3190        53606   
2              3  2024-08-25  2024    8   25  2661.7973        53590   
3              4  2024-08-24  2024    8   24  2661.8805        53590   
4              5  2024-08-23  2024    8   23  2661.9637        53590   
...          ...         ...   ...  ...  ...        ...          ...   
2383        2384  2018-02-17  2018    2   17  1016.2728            2   
2384        2385  2018-02-16  2018    2   16  1016.2871            2   
2385        2386  2018-02-15  2018    2   15  1016.1704            2   
2386        2387  2018-02-14  2018    2   14  1013.2619            2   
2387        2388  2018-02-13  2018    2   13  1003.8325            1   

      activos_totales  activos_neto_totales  acciones_en_circulación  
0        322465959285          2.368288e+11             8.981631

In [19]:
type(RN_info.name)

str